# TAN, SPY, and ICLN daily prices (2016–2024)

Downloads unadjusted daily OHLC and adjusted close from Yahoo Finance via `yfinance` for:

- **TAN** — Invesco Solar ETF
- **SPY** — market control
- **ICLN** — robustness twin

The requested window is January 1, 2016 through December 31, 2024 (inclusive).

In [1]:
# Uncomment if yfinance is not installed in this kernel:
# %pip install -q yfinance

from pathlib import Path

import pandas as pd
import yfinance as yf

TICKERS = ["TAN", "SPY", "ICLN"]
START = "2016-01-01"
END_EXCLUSIVE = "2025-01-01"  # yfinance's end date is exclusive
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)

In [2]:
raw = yf.download(
    TICKERS,
    start=START,
    end=END_EXCLUSIVE,
    interval="1d",
    auto_adjust=False,
    actions=False,
    group_by="ticker",
    progress=False,
    threads=True,
)

if raw.empty:
    raise RuntimeError("Yahoo Finance returned no data.")

raw.head()

Ticker            TAN                                                      \
Price            Open       High        Low      Close  Adj Close  Volume   
Date                                                                        
2016-01-04  30.049999  30.770000  29.620001  30.770000  28.320398  222400   
2016-01-05  31.059999  31.299999  30.219999  30.299999  27.887812  218900   
2016-01-06  29.750000  30.100000  29.459999  29.709999  27.344784  402900   
2016-01-07  28.690001  28.790001  26.830000  26.990000  24.841324  448300   
2016-01-08  27.980000  28.059999  27.200001  27.299999  25.126642  219400   

Ticker      ICLN                                            SPY              \
Price       Open  High   Low Close Adj Close Volume        Open        High   
Date                                                                          
2016-01-04  9.66  9.76  9.52  9.75  8.124291  83700  200.490005  201.029999   
2016-01-05  9.74  9.78  9.64  9.71  8.090960  33800  201.399994  201.899994   
2016-01-06  9.61  9.62  9.52  9.56  7.965971  25400  198.339996  200.059998   
2016-01-07  9.40  9.40  9.21  9.21  7.674330  28000  195.330002  197.440002   
2016-01-08  9.22  9.28  9.11  9.12  7.599336  26800  195.190002  195.850006   

Ticker                                                     
Price              Low       Close   Adj Close     Volume  
Date                                                       
2016-01-04  198.589996  201.020004  169.471558  222353500  
2016-01-05  200.050003  201.360001  169.758194  110845800  
2016-01-06  197.600006  198.820007  167.616867  152112600  
2016-01-07  193.589996  194.050003  163.595398  213436100  
2016-01-08  191.580002  191.919998  161.799728  209817200

In [3]:
required_columns = ["Open", "High", "Low", "Close", "Adj Close"]
frames = []

for ticker in TICKERS:
    ticker_data = raw[ticker][required_columns].copy()
    ticker_data.columns.name = None
    ticker_data = ticker_data.dropna(how="all").reset_index()
    ticker_data.insert(1, "Ticker", ticker)
    frames.append(ticker_data)

prices = (
    pd.concat(frames, ignore_index=True)
    .sort_values(["Ticker", "Date"])
    .reset_index(drop=True)
)

prices

,Date,Ticker,Open,High,Low,Close,Adj Close
0,2016-01-04,ICLN,9.660000,9.760000,9.520000,9.750000,8.124291
1,2016-01-05,ICLN,9.740000,9.780000,9.640000,9.710000,8.090960
2,2016-01-06,ICLN,9.610000,9.620000,9.520000,9.560000,7.965971
3,2016-01-07,ICLN,9.400000,9.400000,9.210000,9.210000,7.674330
4,2016-01-08,ICLN,9.220000,9.280000,9.110000,9.120000,7.599336
...,...,...,...,...,...,...,...
6787,2024-12-24,TAN,34.200001,34.669998,34.029999,34.430000,34.430000
6788,2024-12-26,TAN,34.049999,34.669998,34.040001,34.389999,34.389999
6789,2024-12-27,TAN,34.020000,34.250000,33.650002,33.950001,33.950001
6790,2024-12-30,TAN,33.400002,33.720001,33.119999,33.619999,33.619999


In [4]:
# Validate coverage, uniqueness, and OHLC completeness.
coverage = prices.groupby("Ticker").agg(
    first_date=("Date", "min"),
    last_date=("Date", "max"),
    trading_days=("Date", "size"),
    missing_values=("Open", lambda s: prices.loc[s.index, required_columns].isna().sum().sum()),
)

assert not prices.duplicated(["Ticker", "Date"]).any(), "Duplicate ticker-date rows found."
assert (prices[required_columns].notna().all(axis=1)).all(), "Missing requested price values found."
assert prices["Date"].min() >= pd.Timestamp(START)
assert prices["Date"].max() <= pd.Timestamp("2024-12-31")

coverage

,first_date,last_date,trading_days,missing_values
Ticker,,,,
ICLN,2016-01-04,2024-12-31,2264,0
SPY,2016-01-04,2024-12-31,2264,0
TAN,2016-01-04,2024-12-31,2264,0


In [5]:
# Save one tidy combined file and one file per ETF.
combined_path = OUTPUT_DIR / "tan_spy_icln_daily_2016_2024.csv"
prices.to_csv(combined_path, index=False)

for ticker, ticker_data in prices.groupby("Ticker", sort=False):
    ticker_data.drop(columns="Ticker").to_csv(
        OUTPUT_DIR / f"{ticker.lower()}_daily_2016_2024.csv",
        index=False,
    )

print(f"Saved {len(prices):,} rows to {combined_path.resolve()}")
print("Columns:", prices.columns.tolist())

Saved 6,792 rows to /Users/nick/Desktop/Prediction Markets/data/tan_spy_icln_daily_2016_2024.csv
Columns: ['Date', 'Ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close']


## Polymarket: Trump 2024 election market

Pulls the one-minute-fidelity price history for the **Yes** token in “Will Donald Trump win the 2024 US Presidential Election?” The Gamma API is used to discover the token ID, then the public CLOB `/prices-history` endpoint is queried in 14-day chunks because long high-resolution requests can fail for resolved markets.

In [6]:
import json
import time

import requests

EVENT_SLUG = "presidential-election-winner-2024"
MARKET_SLUG = "will-donald-trump-win-the-2024-us-presidential-election"
GAMMA_URL = "https://gamma-api.polymarket.com/events"
CLOB_HISTORY_URL = "https://clob.polymarket.com/prices-history"

session = requests.Session()
session.headers.update({"User-Agent": "prediction-market-research/1.0"})

/Users/nick/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [7]:
# Discover the resolved market and map its Yes outcome to the CLOB token ID.
response = session.get(GAMMA_URL, params={"slug": EVENT_SLUG}, timeout=30)
response.raise_for_status()
events = response.json()

matches = [
    market
    for event in events
    for market in event.get("markets", [])
    if market.get("slug") == MARKET_SLUG
]
if len(matches) != 1:
    raise RuntimeError(f"Expected one matching market, found {len(matches)}.")

trump_market = matches[0]
outcomes = json.loads(trump_market["outcomes"])
token_ids = json.loads(trump_market["clobTokenIds"])
yes_token_id = dict(zip(outcomes, token_ids))["Yes"]

market_start = pd.Timestamp(trump_market["startDate"]).tz_convert("UTC")
market_end = pd.Timestamp(trump_market["closedTime"]).tz_convert("UTC")

{
    "question": trump_market["question"],
    "condition_id": trump_market["conditionId"],
    "yes_token_id": yes_token_id,
    "market_start": market_start,
    "market_end": market_end,
}

{'question': 'Will Donald Trump win the 2024 US Presidential Election?',
 'condition_id': '0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917',
 'yes_token_id': '21742633143463906290569050155826241533067272736897614950488156847949938836455',
 'market_start': Timestamp('2024-01-04 22:58:00+0000', tz='UTC'),
 'market_end': Timestamp('2024-11-06 15:17:41+0000', tz='UTC')}

In [8]:
def fetch_history_chunk(start, end, max_attempts=4):
    params = {
        "market": yes_token_id,
        "startTs": int(start.timestamp()),
        "endTs": int(end.timestamp()),
        "fidelity": 1,  # minutes
    }
    for attempt in range(max_attempts):
        try:
            response = session.get(CLOB_HISTORY_URL, params=params, timeout=60)
            response.raise_for_status()
            return response.json().get("history", [])
        except requests.RequestException:
            if attempt == max_attempts - 1:
                raise
            time.sleep(2**attempt)


# Explicit date ranges work more reliably than interval="max" for resolved markets.
chunk_size = pd.Timedelta(days=14)
chunk_start = market_start.floor("s")
history = []

while chunk_start < market_end:
    chunk_end = min(chunk_start + chunk_size, market_end)
    chunk = fetch_history_chunk(chunk_start, chunk_end)
    history.extend(chunk)
    print(f"{chunk_start.date()} to {chunk_end.date()}: {len(chunk):,} points")
    chunk_start = chunk_end
    time.sleep(0.1)

if not history:
    raise RuntimeError("The Polymarket CLOB API returned no price history.")

2024-01-04 to 2024-01-18: 20,098 points
2024-01-18 to 2024-02-01: 20,160 points
2024-02-01 to 2024-02-15: 20,160 points
2024-02-15 to 2024-02-29: 20,133 points
2024-02-29 to 2024-03-14: 20,156 points
2024-03-14 to 2024-03-28: 20,160 points
2024-03-28 to 2024-04-11: 20,155 points
2024-04-11 to 2024-04-25: 20,160 points
2024-04-25 to 2024-05-09: 20,139 points
2024-05-09 to 2024-05-23: 20,160 points
2024-05-23 to 2024-06-06: 20,157 points
2024-06-06 to 2024-06-20: 20,160 points
2024-06-20 to 2024-07-04: 20,160 points
2024-07-04 to 2024-07-18: 20,113 points
2024-07-18 to 2024-08-01: 20,155 points
2024-08-01 to 2024-08-15: 20,160 points
2024-08-15 to 2024-08-29: 20,160 points
2024-08-29 to 2024-09-12: 20,160 points
2024-09-12 to 2024-09-26: 20,160 points
2024-09-26 to 2024-10-10: 20,161 points
2024-10-10 to 2024-10-24: 20,160 points
2024-10-24 to 2024-11-06: 18,248 points


In [9]:
trump_prices_1min = (
    pd.DataFrame(history)
    .rename(columns={"t": "timestamp_unix", "p": "price"})
    .drop_duplicates("timestamp_unix", keep="last")
    .sort_values("timestamp_unix")
    .reset_index(drop=True)
)
trump_prices_1min.insert(
    0,
    "timestamp_utc",
    pd.to_datetime(trump_prices_1min["timestamp_unix"], unit="s", utc=True),
)
trump_prices_1min.insert(2, "outcome", "Yes")
trump_prices_1min.insert(3, "token_id", yes_token_id)

assert trump_prices_1min["timestamp_utc"].is_monotonic_increasing
assert trump_prices_1min["timestamp_unix"].is_unique
assert trump_prices_1min["price"].between(0, 1).all()

polymarket_path = OUTPUT_DIR / "polymarket_trump_2024_yes_1min.csv"
trump_prices_1min.to_csv(polymarket_path, index=False)

print(f"Saved {len(trump_prices_1min):,} points to {polymarket_path.resolve()}")
print(
    f"Coverage: {trump_prices_1min['timestamp_utc'].min()} to "
    f"{trump_prices_1min['timestamp_utc'].max()}"
)
trump_prices_1min

Saved 441,431 points to /Users/nick/Desktop/Prediction Markets/data/polymarket_trump_2024_yes_1min.csv
Coverage: 2024-01-04 23:38:02+00:00 to 2024-11-06 15:17:02+00:00


,timestamp_utc,timestamp_unix,outcome,token_id,price
0,2024-01-04 23:38:02+00:00,1704411482,Yes,2174263314346390629056905015582624153306727273...,0.5000
1,2024-01-04 23:39:02+00:00,1704411542,Yes,2174263314346390629056905015582624153306727273...,0.5000
2,2024-01-04 23:40:02+00:00,1704411602,Yes,2174263314346390629056905015582624153306727273...,0.5000
3,2024-01-04 23:41:02+00:00,1704411662,Yes,2174263314346390629056905015582624153306727273...,0.5000
4,2024-01-04 23:42:02+00:00,1704411722,Yes,2174263314346390629056905015582624153306727273...,0.5000
...,...,...,...,...,...
441426,2024-11-06 15:13:02+00:00,1730905982,Yes,2174263314346390629056905015582624153306727273...,0.9985
441427,2024-11-06 15:14:02+00:00,1730906042,Yes,2174263314346390629056905015582624153306727273...,0.9985
441428,2024-11-06 15:15:02+00:00,1730906102,Yes,2174263314346390629056905015582624153306727273...,0.9985
441429,2024-11-06 15:16:02+00:00,1730906162,Yes,2174263314346390629056905015582624153306727273...,0.9975


## Final daily merged dataset

ETF returns are simple returns from adjusted close. `p` is the last Polymarket Yes price observed in each UTC day. The merge retains ETF trading dates covered by the prediction market; `Δp` is then calculated across those retained dates, so a Monday change includes any weekend movement since Friday.

In [ ]:
adjusted_close = (
    prices.pivot(index="Date", columns="Ticker", values="Adj Close")
    .sort_index()
)

etf_returns = adjusted_close.pct_change(fill_method=None).rename(
    columns={ticker: f"{ticker} ret" for ticker in TICKERS}
)

polymarket_daily = (
    trump_prices_1min.assign(
        Date=trump_prices_1min["timestamp_utc"].dt.floor("D").dt.tz_localize(None)
    )
    .groupby("Date")["price"]
    .last()
    .rename("p")
)

merged_daily = etf_returns.join(polymarket_daily, how="inner").sort_index()
merged_daily["Δp"] = merged_daily["p"].diff()
merged_daily = merged_daily[["TAN ret", "SPY ret", "ICLN ret", "p", "Δp"]].dropna()

assert merged_daily.index.is_monotonic_increasing
assert merged_daily.index.is_unique
assert merged_daily.notna().all().all()

merged_path = OUTPUT_DIR / "daily_merged_tan_spy_icln_polymarket_2024.csv"
merged_daily.to_csv(merged_path, index_label="Date")

print(f"Saved {len(merged_daily):,} daily rows to {merged_path.resolve()}")
print(f"Coverage: {merged_daily.index.min().date()} to {merged_daily.index.max().date()}")
merged_daily